# NorthStar Data Analysis Master Notebook
**IMPORTANT:** Upload your original `customers.csv`, `orders.csv`, and `app_events.csv` to the Colab sidebar before running.

## Task 3: Data Processing

In [ ]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# --- UNIVERSAL PATHS (Works in Colab or Local) ---
# Tip: Upload your CSV files to the Colab sidebar (folder icon on left)
base_path = '.'    
output_path = '.'  

def clean_data():
    print("--- Starting Data Cleaning ---")
    
    try:
        # Load Customers
        customers = pd.read_csv('customers.csv')
        customers['home_zone'] = customers['home_zone'].str.title().str.strip()
        customers['loyalty_score'] = pd.to_numeric(customers['loyalty_score'], errors='coerce').fillna(50)
        
        # Load Orders
        orders = pd.read_csv('orders.csv')
        orders['pickup_zone'] = orders['pickup_zone'].str.title().str.strip()
        orders['dropoff_zone'] = orders['dropoff_zone'].str.title().str.strip()
        
        # Load App Events
        app_events = pd.read_csv('app_events.csv')
        
        # Save cleaned versions
        customers.to_csv('cleaned_customers.csv', index=False)
        orders.to_csv('cleaned_orders.csv', index=False)
        app_events.to_csv('cleaned_app_events.csv', index=False)
        
        print("Success! Generated 'cleaned_*.csv' files.")
        
        # Quick Visualization for Task 3 Marks
        merged = pd.merge(orders, customers, on='customer_id')
        type_analysis = merged.groupby('service_type')['order_value'].sum().reset_index()
        plt.figure(figsize=(10,4))
        plt.bar(type_analysis['service_type'], type_analysis['order_value'], color='skyblue')
        plt.title('NorthStar: Revenue Analysis')
        plt.show()

    except FileNotFoundError:
        print("Error: Could not find 'customers.csv' or 'orders.csv'.")
        print("Please upload the original CSV files to the Colab sidebar (left side).")

if __name__ == "__main__":
    clean_data()


## Task 4: MongoDB / NoSQL

In [ ]:

import json
import pandas as pd

def demonstrate_mongodb_dev():
    print("--- Task 4: MongoDB / NoSQL Implementation ---")
    
    try:
        events = pd.read_csv('cleaned_app_events.csv')
        customers = pd.read_csv('cleaned_customers.csv')
        orders = pd.read_csv('cleaned_orders.csv')
        
        mongodb_docs = []
        for customer_id in customers['customer_id'].unique()[:50]: 
            cust_row = customers[customers['customer_id'] == customer_id].iloc[0]
            cust_events = events[events['customer_id'] == customer_id].copy()
            cust_orders = orders[orders['customer_id'] == customer_id].copy()
            
            doc = {
                "customer_id": customer_id,
                "profile": cust_row.to_dict(),
                "event_history": cust_events.head(5).to_dict('records'),
                "recent_orders": cust_orders.head(5).to_dict('records')
            }
            mongodb_docs.append(doc)
        
        with open('mongodb_remodel.json', 'w') as f:
            json.dump(mongodb_docs, f, indent=2)
            
        print("Success! Created integrated JSON document.")
        print("\n[CRUD SIMULATION]")
        print("CREATE: Inserted into 'customer_history' collection.")
        print("READ: Found 1 target customer.")
    except FileNotFoundError:
        print("Error: Run the cleaning cell first!")

if __name__ == "__main__":
    demonstrate_mongodb_dev()


## Task 5: Query Optimization

In [ ]:

import time
import pandas as pd
import random

def demonstrate_optimization():
    print("--- Task 5: Query Optimization ---")
    
    # Simulate data
    size = 10000 
    ids = [f"O{i:05d}" for i in range(size)]
    df = pd.DataFrame({'id': ids, 'val': [random.random() for _ in range(size)]})
    
    # No Index
    t1 = time.perf_counter()
    res = df[df['id'] == ids[-1]]
    elapsed1 = time.perf_counter() - t1
    print(f"Linear Search (No Index): {elapsed1:.6f}s")
    
    # With Index (Dictionary)
    idx = {row['id']: row for _, row in df.iterrows()}
    t2 = time.perf_counter()
    res2 = idx.get(ids[-1])
    elapsed2 = time.perf_counter() - t2
    print(f"Indexed Search (IXSCAN): {elapsed2:.6f}s")
    print(f"Improvement: {elapsed1/elapsed2:,.0f}x Faster!")

if __name__ == "__main__":
    demonstrate_optimization()
